# External Validation: CCRS 2026

Scores the BHPD injury-or-fatal model on 2026 California Crash Reporting System
(CCRS) crashes outside Beverly Hills.

The model expects the engineered training columns (`X.parquet`), so this
notebook builds as many of them as it can directly from CCRS. Any model column
CCRS cannot supply stays NaN. A coverage report shows which columns were
filled before anything is scored.

In [ ]:
import re
import os
import sys
import numpy as np
import pandas as pd
import mlflow

from eda_toolkit import ensure_directory
import model_metrics.model_registry as mr
from model_metrics import (
    show_calibration_curve,
    show_pr_curve,
    show_roc_curve,
    summarize_model_performance,
)

## Set Paths

In [ ]:
mlflow.set_tracking_uri(
    "/home/lshpaner/Python_Projects/bh_traffic/mlruns"
)

from model_metrics import (
    model_registry,
    set_stores,
    best_per_algo,
    load_best_per_algo,
    available,
)

model_registry.configure(root="/home/lshpaner/Python_Projects/bh_traffic")

In [ ]:
sys.path.insert(0, "../bh_traffic")

In [ ]:
data_path = "../data/processed"

In [ ]:
base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data/processed")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function tob ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(image_path_png)
ensure_directory(image_path_svg)


In [ ]:
set_stores("mlruns/models")

In [ ]:
best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")

In [ ]:
available()

In [ ]:
mr.store_summary()

In [ ]:
best = best_per_algo(metric="valid Average Precision")
best

## Model and Training Columns

Load the model the same way as in your evaluation notebook.

In [ ]:
cat_injury_or_fatal = champs["cat_injury_or_fatal"]

In [ ]:
model_cols = pd.read_parquet("../data/processed/X.parquet").columns
print(f"Model expects {len(model_cols)} columns")

## Load and Clean CCRS

- Strip literal `\t` from headers.
- Drop deleted reports; keep the latest version of each crash.
- Remove every Beverly Hills crash so no training rows leak in.

In [ ]:
ext = pd.read_csv("../data/raw/crashes_2026.csv", low_memory=False)
ext.columns = ext.columns.str.replace(r"\\t", "", regex=True).str.strip()

ext = ext[~ext["IsDeleted"].fillna(False).astype(bool)]
ext = ext.sort_values("Report Version").drop_duplicates("Collision Id", keep="last")
ext = ext[ext["City Name"].str.strip().str.upper() != "BEVERLY HILLS"]

dt = pd.to_datetime(ext["Crash Date Time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
ext, dt = ext[dt.notna()], dt[dt.notna()]

y_ext = ((ext["NumberInjured"].fillna(0) + ext["NumberKilled"].fillna(0)) > 0).astype(int)
print(f"Rows: {len(ext):,}  Prevalence: {y_ext.mean():.1%}")

## Build Features

Dummy columns use the same naming rule as training (`<prefix>_<slug>`).
Levels the model never saw go to `other`, blanks to `missing`.

Check the printed column groups first and edit `ENCODE` and the time feature
names so they match your training prefixes.

In [ ]:
## Model column groups, to confirm the prefixes below
prefixes = pd.Series([c.rsplit("_", 1)[0] for c in model_cols]).value_counts()
print(prefixes.head(25))

In [ ]:
def slug(level):
    return re.sub(r"[^0-9a-z]+", "_", str(level).lower()).strip("_") or "blank"

def encode(series, prefix):
    s = series.fillna("MISSING").map(slug)
    levels = {c[len(prefix) + 1:] for c in model_cols if c.startswith(prefix + "_")}
    s = s.where(s.isin(levels), "other")
    return pd.get_dummies(s, prefix=prefix).astype(float)

## CCRS recodes to BHPD values
mult = ext["SecondaryUnitOfMeasure"].map({"F": 1, "M": 5280})
feet_from = (ext["SecondaryDistance"] * mult).fillna(0)
section = (ext["Primary Collision Factor Violation"].str.upper()
           .str.extract(r"(\d{4,5}(?:\([A-Z0-9]+\))*)")[0])
coll_type = ext["Collision Type Description"].replace(
    {"VEHICLE/PEDESTRAIN": "VEHICLE/PEDESTRIAN"})
direction = ext["SecondaryDirection"].map(
    {"N": "NORTH", "S": "SOUTH", "E": "EAST", "W": "WEST"})

## Other Imp Drive: lookup learned from BHPD (PCF section -> label)
bh = pd.read_csv(
    "../data/raw/CollisionRecords_BHPD_Since2015_d_-8252529495950865508.csv",
    low_memory=False)
bh_vc = bh["PCF VC"].str.extract(r"^(\S+)")[0]
vc_to_oid = bh.groupby(bh_vc)["Other Imp Drive"].agg(
    lambda s: s.mode().iat[0] if s.notna().any() else None)
other_imp = section.map(vc_to_oid)

## Edit prefixes to match the groups printed above
ENCODE = {
    "street": ext["PrimaryRoad"],
    "other_imp_drive": other_imp,
    "pcf_section": section,
    "type_of_accident": coll_type,
    "direction_from": direction,
}
dummies = [encode(s, p) for p, s in ENCODE.items()]

hour = dt.dt.hour
numeric = pd.DataFrame({
    "year": dt.dt.year,
    "month": dt.dt.month,
    "weekday": dt.dt.weekday,
    "weekend": (dt.dt.weekday >= 5).astype(int),
    "hour": hour,
    "hour_sin": np.sin(2 * np.pi * hour / 24),
    "hour_cos": np.cos(2 * np.pi * hour / 24),
    "night": ((hour >= 20) | (hour < 6)).astype(int),
    "am_rush": hour.between(7, 9).astype(int),
    "pm_rush": hour.between(16, 19).astype(int),
    "feet_from": feet_from,
    "intersection": (feet_from == 0).astype(int),
}, index=ext.index)

feats = pd.concat([numeric, *dummies], axis=1)

## Align to the Model

Any model column CCRS could not supply becomes NaN. Party features (party
type, involved party, collision partner, VRU flags) need the CCRS parties file,
so they stay NaN in this version.

In [ ]:
X_ext = feats.reindex(columns=model_cols)

filled = X_ext.notna().any()
print(f"Columns with data: {filled.sum()} of {len(model_cols)}")
print("Still NaN:", sorted(filled[~filled].index)[:40])

## Score

In [ ]:
summarize_model_performance(model=[cat_injury_or_fatal], X=X_ext, y=y_ext)

In [ ]:
show_roc_curve(model=[cat_injury_or_fatal], X=X_ext, y=y_ext)
show_pr_curve(model=[cat_injury_or_fatal], X=X_ext, y=y_ext)
show_calibration_curve(model=[cat_injury_or_fatal], X=X_ext, y=y_ext)

### Calibration Intercept and Slope

Intercept away from 0 means the overall risk level shifted; slope below 1
means predictions are too extreme.

In [ ]:
import statsmodels.api as sm

p = cat_injury_or_fatal.predict_proba(X_ext)[:, 1].clip(1e-6, 1 - 1e-6)
lp = np.log(p / (1 - p))
slope = sm.Logit(y_ext, sm.add_constant(lp)).fit(disp=0).params[1]
citl = sm.GLM(y_ext, np.ones_like(lp), offset=lp,
              family=sm.families.Binomial()).fit().params[0]
print(f"Calibration-in-the-large: {citl:.2f}  Slope: {slope:.2f}")

## Notes

- **Prevalence differs.** CCRS includes routine property-damage-only crashes,
  so injury prevalence is far below BHPD's ~66%. Compare AUC directly; expect
  the 0.11 threshold and calibration not to transfer.
- **Case mix can raise AUC.** Easy negatives (minor rear-ends) make
  discrimination easier, so a higher AUC than internal is not a better model.
- **Street dummies** are Beverly Hills streets and will mostly land in `other`.
- **Party features are NaN** until the parties file is joined; internally those
  were worth about 0.07 AUC.